Transformación datos de recaudos bronze a capa silver. -- 07/11/2026

Este notebook tendrá las siguientes responsabilidades:
1. Leer la tabla de recaudos de la capa bronze
2. Realizar limpieza y validacion de los datos por medio de las siguientes reglas de negocio de Aguas de Manizales
3. Escribir los registros válidos en silver.recaudos_transformed y los rechazados en silver.recaudos_rechazados



 

Reglas aplicadas a los registros de recaudos:

PERIODO: nulo, formato distinto a yyyyMM (mes 01-12) o periodo futuro (se aceptan periodos antiguos: pagos de cartera vencida)

SUSCRIPTOR: nulo o fuera del rango 0 - 200.000 (suscriptores actuales de Aguas de Manizales; excluye cuentas de prueba como 1111111111)

ENTIDAD: código o descripción de la entidad recaudadora nula o vacía

PUNTO DE RECAUDO: código nulo
FECHA DE PAGO: nula, futura o anterior al inicio del periodo pagado

TARIFA / CONCEPTO: código de tarifa nulo, código o descripción de concepto nula
VALOR PAGADO ($): nulo, cero o negativo (reversiones/ajustes)
DUPLICADOS: registros idénticos en todas las columnas de negocio (se conserva uno)

Advertencia (no se rechaza): valor pagado atípico, mayor a Q3 + 3·IQR de su código de tarifa. Corresponde en su mayoría a abonos de cartera o grandes clientes, por lo que rechazarlos distorsionaría el total recaudado.

Los registros válidos van a la capa silver, tabla silver.recaudos_transformed
Los registros rechazados van a la capa silver, tabla silver.recaudos_rechazados con sus motivos para revisión por parte de los analistas de negocio de AGM.


In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F  
from pyspark.sql import Window

In [0]:
dbutils.widgets.text("catalogo", "catalog_pysmartdata")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_bronze_rec = spark.table(f"{catalogo}.{esquema_source}.recaudos")

# Parámetros de calidad
SUSCRIPTOR_MIN, SUSCRIPTOR_MAX = 0, 200000    ## de acuerdo al No. de suscriptores actuales de AGM
FACTOR_IQR = 3.0         # 3 = solo atípicos extremos (advertencia, no rechazo)

# Límite superior de valor pagado atípico por código de tarifa (IQR sobre pagos positivos)
limites_tarifa = (
    df_bronze_rec
    .filter(F.col("valor_pagado") > 0)
    .groupBy("codigo_tarifa")
    .agg(F.percentile_approx("valor_pagado", 0.25, 10000).alias("q1"),
         F.percentile_approx("valor_pagado", 0.75, 10000).alias("q3"),
         F.count("*").alias("pagos"))
    .withColumn("limite_sup_valor", F.col("q3") + FACTOR_IQR * (F.col("q3") - F.col("q1")))
)
display(limites_tarifa.orderBy(F.desc("pagos")))

In [0]:
cols_negocio = [c for c in df_bronze_rec.columns if c != "ingestion_date"]
w_dup = Window.partitionBy(*cols_negocio).orderBy(F.col("ingestion_date").desc())

periodo_str = F.col("periodo").cast("string")
inicio_periodo = F.to_date(periodo_str, "yyyyMM")

def vacio(c):
    return F.col(c).isNull() | (F.trim(F.col(c)) == "")

reglas = {
    "PERIODO_NULO":                   F.col("periodo").isNull(),
    "PERIODO_FORMATO_INVALIDO":       F.col("periodo").isNotNull() & ~periodo_str.rlike(r"^(19|20)[0-9]{2}(0[1-9]|1[0-2])$"),
    "PERIODO_FUTURO":                 F.col("periodo") > F.date_format(F.current_date(), "yyyyMM").cast("int"),
    "SUSCRIPTOR_NULO":                F.col("suscriptor").isNull(),
    "SUSCRIPTOR_FUERA_DE_RANGO":      ~F.col("suscriptor").between(SUSCRIPTOR_MIN, SUSCRIPTOR_MAX),
    "ENTIDAD_NULA":                   vacio("entidad") | vacio("descripcion"),
    "PUNTO_RECAUDO_NULO":             F.col("codigo_punto_recaudo").isNull(),
    "FECHA_PAGO_NULA":                F.col("fecha_pago").isNull(),
    "FECHA_PAGO_FUTURA":              F.col("fecha_pago") > F.current_date(),
    "FECHA_PAGO_ANTERIOR_AL_PERIODO": F.col("fecha_pago") < inicio_periodo,
    "TARIFA_NULA":                    F.col("codigo_tarifa").isNull(),
    "CONCEPTO_NULO":                  F.col("codigo_concepto").isNull() | vacio("concepto"),
    "VALOR_PAGADO_NULO":              F.col("valor_pagado").isNull(),
    "VALOR_PAGADO_CERO_O_NEGATIVO":   F.col("valor_pagado") <= 0,
    "REGISTRO_DUPLICADO":             F.col("_rn") > 1,
}

df_validado = (
    df_bronze_rec
    .withColumn("_rn", F.row_number().over(w_dup))
    .withColumn(
        "motivos_rechazo",
        F.filter(F.array(*[F.when(cond, F.lit(nombre)) for nombre, cond in reglas.items()]),
                 lambda x: x.isNotNull())
    )
    .withColumn("es_valido", F.size("motivos_rechazo") == 0)
    # Advertencia (no se rechaza): valor pagado atípico frente a su tarifa
    .join(F.broadcast(limites_tarifa.select("codigo_tarifa", "limite_sup_valor")), "codigo_tarifa", "left")
    .withColumn("es_valor_atipico", F.coalesce(F.col("valor_pagado") > F.col("limite_sup_valor"), F.lit(False)))
)

In [0]:
total_registros = df_validado.count()
print(f"Total registros bronze: {total_registros}")

display(df_validado.groupBy("es_valido").count())

# Registros por motivo de rechazo (un registro puede tener varios motivos)
display(
    df_validado
    .select(F.explode("motivos_rechazo").alias("motivo"))
    .groupBy("motivo").count()
    .withColumn("pct", F.round(F.col("count") * 100 / total_registros, 4))
    .orderBy(F.desc("count"))
)

# Advertencia (no se rechaza): valor pagado atípico frente a su tarifa
print("Registros válidos con valor pagado atípico (advertencia):",
      df_validado.filter(F.col("es_valido") & F.col("es_valor_atipico")).count())

In [0]:
tabla_silver = f"{catalogo}.{esquema_sink}.recaudos_transformed"
tabla_rechazos = f"{catalogo}.{esquema_sink}.recaudos_rechazados"

# Válidos: normalizar textos y respetar el orden de columnas de la tabla silver
df_validos = (
    df_validado
    .filter(F.col("es_valido"))
    .withColumn("descripcion", F.trim(F.regexp_replace(F.col("descripcion"), r"\s+", " ")))
    .withColumn("descripcion_punto_recaudo", F.trim(F.col("descripcion_punto_recaudo")))
    .withColumn("concepto", F.trim(F.col("concepto")))
    .select(spark.table(tabla_silver).columns)
)

# Rechazados: se conservan con sus motivos para revisión y reproceso
df_rechazados = (
    df_validado
    .filter(~F.col("es_valido"))
    .drop("_rn", "es_valido", "limite_sup_valor", "es_valor_atipico")
    .withColumn("fecha_validacion", F.current_timestamp())
    .select(spark.table(tabla_rechazos).columns)
)

In [0]:
df_validos.write.mode("overwrite").insertInto(tabla_silver)

df_rechazados.write.mode("overwrite").insertInto(tabla_rechazos)

In [0]:
n_silver = spark.table(tabla_silver).count()
n_rech = spark.table(tabla_rechazos).count()
print(f"Bronze: {total_registros} | registros silver válidos: {n_silver} | registros silver rechazados: {n_rech}")
assert n_silver + n_rech == total_registros, "Los conteos no cuadran"

In [0]:
# Conciliación: recaudo bronze vs recaudo válido en silver por periodo de pago
display(
    spark.table(f"{catalogo}.{esquema_source}.recaudos")
    .groupBy(F.date_format("fecha_pago", "yyyy-MM").alias("mes_pago"))
    .agg(F.count("*").alias("pagos_bronze"), F.round(F.sum("valor_pagado"), 0).alias("valor_bronze"))
    .join(
        spark.table(tabla_silver)
        .groupBy(F.date_format("fecha_pago", "yyyy-MM").alias("mes_pago"))
        .agg(F.count("*").alias("pagos_silver"), F.round(F.sum("valor_pagado"), 0).alias("valor_silver")),
        "mes_pago", "left")
    .orderBy("mes_pago")
)

In [0]:
%sql
SELECT *
FROM catalog_pysmartdata.silver.recaudos_transformed
LIMIT 100;

In [0]:
%sql
SELECT periodo, suscriptor, descripcion AS entidad, fecha_pago, codigo_tarifa, valor_pagado,
       array_join(motivos_rechazo, ', ') AS motivos, fecha_validacion
FROM catalog_pysmartdata.silver.recaudos_rechazados
ORDER BY periodo, suscriptor;

In [0]:
%sql
SELECT motivo, COUNT(*) AS registros
FROM catalog_pysmartdata.silver.recaudos_rechazados
LATERAL VIEW explode(motivos_rechazo) t AS motivo
GROUP BY motivo
ORDER BY registros DESC;